# BitBrains — Workload & Resource Telemetry EDA

## Objective

This notebook performs exploratory data analysis on the BitBrains
workload/resource telemetry dataset.

The objectives are to:

- Understand the structure and schema of the BitBrains dataset
- Inspect VM workload activity
- Analyze CPU, memory, disk, and network utilization
- Validate timestamps and telemetry records
- Identify missing and duplicated observations
- Analyze VM/resource behavior over time
- Construct workload-level summary features
- Produce an analysis-ready telemetry dataset

## Role in the FinOps Platform

BitBrains is treated as an auxiliary workload/resource telemetry
dataset.

It is NOT used as the primary financial ground truth.

FOCUS 1.0 provides actual cloud expenditure, while BitBrains
provides workload and resource utilization context.

## Resources

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

BITBRAINS_DIR = DRIVE_ROOT / "fastStorage" / "2013-8"

print("BitBrains directory:")
print(BITBRAINS_DIR)

print("\nExists:", BITBRAINS_DIR.exists())

BitBrains directory:
/content/drive/MyDrive/fastStorage/2013-8

Exists: True


In [ ]:
RAW_DIR = BITBRAINS_DIR

In [ ]:
uploaded = {}
COLAB_ROOT = Path('/content')

In [ ]:
bitbrains_files = sorted(
    BITBRAINS_DIR.glob("*.csv")
)

print("Total CSV files:", len(bitbrains_files))

Total CSV files: 1250


In [ ]:
total_size_gb = sum(
    file.stat().st_size for file in bitbrains_files
) / (1024 ** 3)

print(f"Total CSV files: {len(bitbrains_files)}")
print(f"Total size: {total_size_gb:.2f} GB")

Total CSV files: 1250
Total size: 1.16 GB


In [ ]:
sample_files = [
    bitbrains_files[0],
    bitbrains_files[len(bitbrains_files) // 2],
    bitbrains_files[-1]
]

for file in sample_files:
    print("\n" + "=" * 80)
    print("FILE:", file.name)

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(3):
            print(f.readline().strip())


FILE: 1.csv
Timestamp [ms];	CPU cores;	CPU capacity provisioned [MHZ];	CPU usage [MHZ];	CPU usage [%];	Memory capacity provisioned [KB];	Memory usage [KB];	Disk read throughput [KB/s];	Disk write throughput [KB/s];	Network received throughput [KB/s];	Network transmitted throughput [KB/s]
1376314846;	4;	11703.99824;	10912.027692426667;	93.23333333333333;	6.7108864E7;	6129274.4;	0.13333333333333333;	15981.6;	0.0;	2.1333333333333333
1376315146;	4;	11703.99824;	10890.57036232;	93.05;	6.7108864E7;	6755624.0;	1.3333333333333333;	19137.333333333332;	0.0;	2.6

FILE: 436.csv
Timestamp [ms];	CPU cores;	CPU capacity provisioned [MHZ];	CPU usage [MHZ];	CPU usage [%];	Memory capacity provisioned [KB];	Memory usage [KB];	Disk read throughput [KB/s];	Disk write throughput [KB/s];	Network received throughput [KB/s];	Network transmitted throughput [KB/s]
1376314846;	4;	10399.997204;	168.13328813133333;	1.6166666666666665;	2185216.0;	123032.8;	0.0;	12.733333333333334;	4.8;	0.0
1376315146;	4;	10399.9972

In [ ]:
file_sizes = pd.DataFrame({
    "file": [f.name for f in bitbrains_files],
    "size_mb": [
        f.stat().st_size / (1024 ** 2)
        for f in bitbrains_files
    ]
})

display(file_sizes.describe())

display(
    file_sizes.sort_values(
        "size_mb",
        ascending=False
    ).head(10)
)

,size_mb
count,1250.000000
mean,0.951066
std,0.234934
min,0.000781
25%,0.876549
50%,0.998173
75%,1.079884
max,1.755983


,file,size_mb
818,61.csv,1.755983
951,73.csv,1.522959
824,615.csv,1.439100
816,608.csv,1.431204
539,359.csv,1.418095
867,654.csv,1.416502
744,543.csv,1.410835
1076,842.csv,1.405258
535,355.csv,1.402313
1037,807.csv,1.399149


In [ ]:
import shutil

for filename in uploaded.keys():
    source = COLAB_ROOT / filename
    destination = RAW_DIR / filename

    shutil.move(
        str(source),
        str(destination)
    )

print("Files in BitBrains raw directory:")

for file in RAW_DIR.iterdir():
    print(" -", file.name)

Files in BitBrains raw directory:
 - 1234.csv
 - 138.csv
 - 1207.csv
 - 1250.csv
 - 1241.csv
 - 1239.csv
 - 1202.csv
 - 139.csv
 - 143.csv
 - 1201.csv
 - 1194.csv
 - 1186.csv
 - 1205.csv
 - 120.csv
 - 1218.csv
 - 1198.csv
 - 118.csv
 - 1178.csv
 - 1182.csv
 - 1188.csv
 - 1220.csv
 - 1174.csv
 - 1230.csv
 - 1222.csv
 - 1181.csv
 - 146.csv
 - 1203.csv
 - 140.csv
 - 1210.csv
 - 13.csv
 - 1180.csv
 - 1197.csv
 - 1217.csv
 - 127.csv
 - 1209.csv
 - 141.csv
 - 1196.csv
 - 131.csv
 - 1192.csv
 - 1216.csv
 - 1248.csv
 - 1223.csv
 - 1229.csv
 - 1221.csv
 - 1191.csv
 - 137.csv
 - 14.csv
 - 1185.csv
 - 1245.csv
 - 1214.csv
 - 1183.csv
 - 1228.csv
 - 135.csv
 - 1179.csv
 - 227.csv
 - 196.csv
 - 219.csv
 - 199.csv
 - 156.csv
 - 237.csv
 - 203.csv
 - 158.csv
 - 224.csv
 - 172.csv
 - 232.csv
 - 19.csv
 - 211.csv
 - 230.csv
 - 168.csv
 - 175.csv
 - 159.csv
 - 162.csv
 - 209.csv
 - 205.csv
 - 214.csv
 - 220.csv
 - 226.csv
 - 216.csv
 - 223.csv
 - 148.csv
 - 15.csv
 - 151.csv
 - 204.csv
 - 234.csv
 - 194

In [ ]:
bitbrains_files = list(RAW_DIR.iterdir())

print(f"Files found: {len(bitbrains_files)}\n")

for file in bitbrains_files:
    print(
        f"{file.name:<50} "
        f"{file.stat().st_size / (1024**2):.2f} MB"
    )

Files found: 1250

1234.csv                                           0.73 MB
138.csv                                            1.02 MB
1207.csv                                           0.84 MB
1250.csv                                           0.50 MB
1241.csv                                           0.50 MB
1239.csv                                           0.50 MB
1202.csv                                           0.87 MB
139.csv                                            1.03 MB
143.csv                                            1.11 MB
1201.csv                                           0.68 MB
1194.csv                                           0.67 MB
1186.csv                                           0.79 MB
1205.csv                                           0.94 MB
120.csv                                            1.01 MB
1218.csv                                           1.02 MB
1198.csv                                           0.73 MB
118.csv                              

In [ ]:
bitbrains_files = list(RAW_DIR.iterdir())

print(f"Files found: {len(bitbrains_files)}\n")

for i, file in enumerate(bitbrains_files, start=1):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{i}. {file.name} — {size_mb:.2f} MB")

Files found: 1250

1. 1234.csv — 0.73 MB
2. 138.csv — 1.02 MB
3. 1207.csv — 0.84 MB
4. 1250.csv — 0.50 MB
5. 1241.csv — 0.50 MB
6. 1239.csv — 0.50 MB
7. 1202.csv — 0.87 MB
8. 139.csv — 1.03 MB
9. 143.csv — 1.11 MB
10. 1201.csv — 0.68 MB
11. 1194.csv — 0.67 MB
12. 1186.csv — 0.79 MB
13. 1205.csv — 0.94 MB
14. 120.csv — 1.01 MB
15. 1218.csv — 1.02 MB
16. 1198.csv — 0.73 MB
17. 118.csv — 0.99 MB
18. 1178.csv — 0.79 MB
19. 1182.csv — 0.78 MB
20. 1188.csv — 1.09 MB
21. 1220.csv — 0.97 MB
22. 1174.csv — 0.79 MB
23. 1230.csv — 1.13 MB
24. 1222.csv — 0.97 MB
25. 1181.csv — 0.79 MB
26. 146.csv — 1.11 MB
27. 1203.csv — 0.92 MB
28. 140.csv — 1.02 MB
29. 1210.csv — 0.47 MB
30. 13.csv — 0.71 MB
31. 1180.csv — 0.79 MB
32. 1197.csv — 0.81 MB
33. 1217.csv — 0.78 MB
34. 127.csv — 1.02 MB
35. 1209.csv — 0.01 MB
36. 141.csv — 1.09 MB
37. 1196.csv — 0.94 MB
38. 131.csv — 1.12 MB
39. 1192.csv — 0.77 MB
40. 1216.csv — 0.73 MB
41. 1248.csv — 0.50 MB
42. 1223.csv — 0.40 MB
43. 1229.csv — 1.12 MB
44. 1221.csv 

In [ ]:
for file in bitbrains_files:
    print(file.name, "→", file.suffix.lower())

1.csv → .csv
10.csv → .csv
100.csv → .csv
1000.csv → .csv
1001.csv → .csv
1002.csv → .csv
1003.csv → .csv
1004.csv → .csv
1005.csv → .csv
1006.csv → .csv
1007.csv → .csv
1008.csv → .csv
1009.csv → .csv
101.csv → .csv
1010.csv → .csv
1011.csv → .csv
1012.csv → .csv
1013.csv → .csv
1014.csv → .csv
1015.csv → .csv
1016.csv → .csv
1017.csv → .csv
1018.csv → .csv
1019.csv → .csv
102.csv → .csv
1020.csv → .csv
1021.csv → .csv
1022.csv → .csv
1023.csv → .csv
1024.csv → .csv
1025.csv → .csv
1026.csv → .csv
1027.csv → .csv
1028.csv → .csv
1029.csv → .csv
103.csv → .csv
1030.csv → .csv
1031.csv → .csv
1032.csv → .csv
1033.csv → .csv
1034.csv → .csv
1035.csv → .csv
1036.csv → .csv
1037.csv → .csv
1038.csv → .csv
1039.csv → .csv
104.csv → .csv
1040.csv → .csv
1041.csv → .csv
1042.csv → .csv
1043.csv → .csv
1044.csv → .csv
1045.csv → .csv
1046.csv → .csv
1047.csv → .csv
1048.csv → .csv
1049.csv → .csv
105.csv → .csv
1050.csv → .csv
1051.csv → .csv
1052.csv → .csv
1053.csv → .csv
1054.csv → .csv
105

In [ ]:
csv_files = [
    f for f in bitbrains_files
    if f.suffix.lower() == ".csv"
]

if csv_files:
    for file in csv_files:
        print(f"\n===== {file.name} =====")

        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for _ in range(5):
                print(f.readline().strip())
else:
    print("No CSV files found.")

Streaming output truncated to the last 5000 lines.
1376315446;	1;	2599.99945;	19.06666263333333;	0.7333333333333333;	4194304.0;	106254.13333333333;	0.0;	1.0;	0.06666666666666667;	0.2
1376315746;	1;	2599.99945;	17.333329666666668;	0.6666666666666667;	4194304.0;	156586.13333333333;	0.0;	1.2;	0.0;	0.06666666666666667

===== 356.csv =====
Timestamp [ms];	CPU cores;	CPU capacity provisioned [MHZ];	CPU usage [MHZ];	CPU usage [%];	Memory capacity provisioned [KB];	Memory usage [KB];	Disk read throughput [KB/s];	Disk write throughput [KB/s];	Network received throughput [KB/s];	Network transmitted throughput [KB/s]
1376314846;	1;	2599.999334;	17.333328893333338;	0.6666666666666667;	4194304.0;	176157.86666666667;	0.0;	1.4666666666666666;	0.06666666666666667;	0.0
1376315146;	1;	2599.999334;	13.866663114666666;	0.5333333333333333;	4194304.0;	78291.46666666666;	0.0;	1.3333333333333333;	0.0;	0.0
1376315446;	1;	2599.999334;	15.599996004;	0.6;	4194304.0;	95069.06666666667;	0.0;	1.4666666666666666;	0.0

In [ ]:
text_files = [
    f for f in bitbrains_files
    if f.suffix.lower() in [".txt", ".tsv"]
]

if text_files:
    for file in text_files:
        print(f"\n===== {file.name} =====")

        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for _ in range(10):
                print(f.readline().strip())
else:
    print("No TXT/TSV files found.")

No TXT/TSV files found.


In [ ]:
if not csv_files:
    raise FileNotFoundError("No CSV file found.")

if len(csv_files) == 1:
    BITBRAINS_FILE = csv_files[0]
else:
    print("Multiple CSV files found:")
    for i, file in enumerate(csv_files):
        print(i, file.name)

    # Change this index if needed
    BITBRAINS_FILE = csv_files[0]

print("Loading:", BITBRAINS_FILE.name)

bitbrains_df = pd.read_csv(
    BITBRAINS_FILE,
    low_memory=False
)

print("Loaded successfully.")
print("Shape:", bitbrains_df.shape)

Multiple CSV files found:
0 1.csv
1 10.csv
2 100.csv
3 1000.csv
4 1001.csv
5 1002.csv
6 1003.csv
7 1004.csv
8 1005.csv
9 1006.csv
10 1007.csv
11 1008.csv
12 1009.csv
13 101.csv
14 1010.csv
15 1011.csv
16 1012.csv
17 1013.csv
18 1014.csv
19 1015.csv
20 1016.csv
21 1017.csv
22 1018.csv
23 1019.csv
24 102.csv
25 1020.csv
26 1021.csv
27 1022.csv
28 1023.csv
29 1024.csv
30 1025.csv
31 1026.csv
32 1027.csv
33 1028.csv
34 1029.csv
35 103.csv
36 1030.csv
37 1031.csv
38 1032.csv
39 1033.csv
40 1034.csv
41 1035.csv
42 1036.csv
43 1037.csv
44 1038.csv
45 1039.csv
46 104.csv
47 1040.csv
48 1041.csv
49 1042.csv
50 1043.csv
51 1044.csv
52 1045.csv
53 1046.csv
54 1047.csv
55 1048.csv
56 1049.csv
57 105.csv
58 1050.csv
59 1051.csv
60 1052.csv
61 1053.csv
62 1054.csv
63 1055.csv
64 1056.csv
65 1057.csv
66 1058.csv
67 1059.csv
68 106.csv
69 1060.csv
70 1061.csv
71 1062.csv
72 1063.csv
73 1064.csv
74 1065.csv
75 1066.csv
76 1067.csv
77 1068.csv
78 1069.csv
79 107.csv
80 1070.csv
81 1071.csv
82 1072.csv
8

In [ ]:
display(bitbrains_df.head())

,Timestamp [ms];\tCPU cores;\tCPU capacity provisioned [MHZ];\tCPU usage [MHZ];\tCPU usage [%];\tMemory capacity provisioned [KB];\tMemory usage [KB];\tDisk read throughput [KB/s];\tDisk write throughput [KB/s];\tNetwork received throughput [KB/s];\tNetwork transmitted throughput [KB/s]
0,1376314846;\t4;\t11703.99824;\t10912.027692426...
1,1376315146;\t4;\t11703.99824;\t10890.57036232;...
2,1376315446;\t4;\t11703.99824;\t10434.11443096;...
3,1376315746;\t4;\t11703.99824;\t10539.450415120...
4,1376316046;\t4;\t11703.99824;\t10951.041019893...


In [ ]:
# Reload BitBrains using the correct delimiter
BITBRAINS_FILE = bitbrains_files[0]

print("Loading:", BITBRAINS_FILE.name)

bitbrains_df = pd.read_csv(
    BITBRAINS_FILE,
    sep=";\t",
    engine="python"
)

print("Loaded successfully.")
print("Shape:", bitbrains_df.shape)

Loading: 1234.csv
Loaded successfully.
Shape: (5754, 11)
